# 15b — Stronger multi-detector ensemble for Kaggle (Google Colab)

**Target:** improve the previous public score **0.20783** using checkpoints already trained for HW2. We **cannot guarantee** exceeding the organizer baseline **0.22163**; the private and public test distributions are unknown.

**Default: no training, no downloading pretrained models.** Use three RT-DETR-L CV checkpoints, three tuned RetinaNet CV checkpoints, and, if available, the final RetinaNet trained on all 56 training videos. **Optional:** add all three tuned Faster R-CNN checkpoints (up to 10 trained models in the ensemble); disabled by default to save inference time. Each checkpoint is distinct even if architecture repeats.

**Evaluation:** tune a small set of class-aware Weighted Boxes Fusion (WBF) settings on **four whole validation videos**; report an untouched check on **three other validation videos**. The test set is used strictly for inference.

**Persistence:** predictions, chosen fusion parameters, evaluations, and `submission.csv` go directly to `MyDrive/UCU_AI_HW2/ensemble15_final`. To reproduce prior Kaggle result quickly, upload completed `val_step3_*.json.gz` and `test_all_*.json.gz` caches to `MyDrive/UCU_AI_HW2/ensemble_cache` (or a ZIP `ensemble_cache.zip`).

**Important:** 3-fold CV checkpoint ensembling is not the same as independently training every model on the full 56-video training set. New full-training RetinaNet is included when available. Optional RT-DETR fine-tuning is a separate experiment, not part of the default submission.


## 1 — Required files on Google Drive

Under `MyDrive/UCU_AI_HW2`:

**Required:**
- `visdrone_competition.zip` — the competition data with `train/annotations_coco.json`, `val/annotations_coco.json`, and `test/images`.
- `rtdetr_tuning_essentials.zip` — existing RT-DETR-L `lr2e4_fold_0/1/2/weights/best.pt`.
- RetinaNet tuned weights: `retinanet_v2_tuning/resolution_768/fold_0/latest.pt`, `fold_1/latest.pt`, `fold_2/latest.pt`, **or** completed full-training weight `retinanet_final_6ep/best.pt` (the notebook can run with only one of these sources, though 3-fold CV is needed to recreate the earlier ensemble).

**Optional:**
- `retinanet_final_6ep/best.pt` if full-training RetinaNet has finished — adds a distinct source.
- 3 Faster R-CNN tuned checkpoints under `faster_rcnn_tuning/` — turn on `INCLUDE_FASTER` later for an optional 10-model experiment.
- `ensemble_cache.zip` or `ensemble_cache/` with old Kaggle prediction JSON.GZ files. This can save hours of GPU inference. **Do not rename cached predictions to make them appear to correspond to a different checkpoint.**

You do **not** need `cv_splits_step6.csv`, YOLO weights, or old submissions. Use a GPU when prediction caches are missing.


In [1]:
%pip -q install "ultralytics==8.4.204" pycocotools

In [2]:
import os, gc, io, re, sys, json, csv, gzip, math, time, shutil, zipfile, hashlib, random
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, torch, torchvision
from PIL import Image
from tqdm.auto import tqdm
from torchvision.transforms.functional import pil_to_tensor
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import RTDETR
from google.colab import drive

drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive/UCU_AI_HW2')
OUT=DRIVE/'ensemble15_final'; OUT.mkdir(parents=True,exist_ok=True)
LOCAL=Path('/content/ensemble15');LOCAL.mkdir(exist_ok=True)
CKPTS=LOCAL/'checkpoints';CKPTS.mkdir(exist_ok=True)
CACHE=DRIVE/'ensemble_cache';CACHE.mkdir(exist_ok=True)
COMPETITION_ZIP=DRIVE/'visdrone_competition.zip'
RT_ARCHIVE=DRIVE/'rtdetr_tuning_essentials.zip'
RET_FINAL=DRIVE/'retinanet_final_6ep/best.pt'
assert zipfile.is_zipfile(COMPETITION_ZIP), f'Missing {COMPETITION_ZIP}'
assert zipfile.is_zipfile(RT_ARCHIVE), f'Missing {RT_ARCHIVE}'
INCLUDE_FASTER=False  # Optional: requires 3 tuned Faster R-CNN weights; adds 3 more sources.
DEVICE='cuda:0' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE, 'Drive:',DRIVE)
print('Final RetinaNet:',RET_FINAL.exists())
print('Prediction caches:',len(list(CACHE.glob('*.json.gz'))))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Mounted at /content/drive
Device: cuda:0 Drive: /content/drive/MyDrive/UCU_AI_HW2
Final RetinaNet: True
Prediction caches: 0


## 2 — Restore weights and annotations without extracting the full 11-GB dataset

This is faster than unzipping all images when previously saved inference caches are available. When missing caches require inference, the whole competition ZIP will be extracted later **only on demand**.

In [3]:
# Extract exactly the RT-DETR lr2e4 best.pt from each fold (no pretrained download).
RT_WEIGHTS={}
with zipfile.ZipFile(RT_ARCHIVE) as z:
    for fold in (0,1,2):
        members=[m for m in z.namelist() if m.endswith(f'lr2e4_fold_{fold}/weights/best.pt') or
                 re.search(fr'(^|/)lr2e4_fold_{fold}/weights/best\.pt$',m)]
        assert len(members)==1, f'Fold {fold}: checkpoint matches {members}'
        destination=CKPTS/f'rtdetr_lr2e4_fold_{fold}.pt'
        if not destination.is_file():
            with z.open(members[0]) as src, destination.open('wb') as dst: shutil.copyfileobj(src,dst)
        assert destination.stat().st_size>1_000_000
        RT_WEIGHTS[fold]=destination

# Optional CV RetinaNet checkpoints, used if the final 56-video RetinaNet is missing.
RET_CV={}
for fold in (0,1,2):
    match=DRIVE/f'retinanet_v2_tuning/resolution_768/fold_{fold}/latest.pt'
    if match.is_file():RET_CV[fold]=match
assert RET_FINAL.is_file() or len(RET_CV)==3, (
    'Need retinanet_final_6ep/best.pt OR all three resolution_768/fold_{0,1,2}/latest.pt checkpoints.'
)
WEIGHTS={**{f'rt_f{fold}':('rt',p) for fold,p in RT_WEIGHTS.items()}}
if RET_FINAL.is_file(): WEIGHTS['retina_final']=('retina',RET_FINAL)
if len(RET_CV)==3: WEIGHTS.update({f'retina_f{fold}':('retina',p) for fold,p in RET_CV.items()})

FASTER_FOLD_PATHS={
  0: DRIVE/'faster_rcnn_tuning/resolution_768/latest.pt',
  1: DRIVE/'faster_rcnn_tuning/resolution_768_3fold/fold_1/latest.pt',
  2: DRIVE/'faster_rcnn_tuning/resolution_768_3fold/fold_2/latest.pt',
}
if INCLUDE_FASTER:
    for fold,p in FASTER_FOLD_PATHS.items():
        assert p.is_file(), f'Faster R-CNN checkpoint missing: {p}'
        WEIGHTS[f'faster_f{fold}']=('faster',p)
for name,(kind,p) in WEIGHTS.items():print(name,kind,round(p.stat().st_size/1e6,1),'MB')

# Read COCO holdout JSON, without loading gigabytes of images.
with zipfile.ZipFile(COMPETITION_ZIP) as z:
    val_names=[n for n in z.namelist() if n.endswith('val/annotations_coco.json')]
    train_names=[n for n in z.namelist() if n.endswith('train/annotations_coco.json')]
    assert len(val_names)==len(train_names)==1,(val_names,train_names)
    VAL_COCO=json.loads(z.read(val_names[0]))
    TRAIN_COCO=json.loads(z.read(train_names[0]))
    print('Dataset train',len(TRAIN_COCO['images']),'val',len(VAL_COCO['images']))
assert len(TRAIN_COCO['images'])==24201 and len(VAL_COCO['images'])==2846
VAL_JSON=LOCAL/'val_coco.json';VAL_JSON.write_text(json.dumps(VAL_COCO))
GT=COCO(str(VAL_JSON))
meta=pd.DataFrame([{'id':int(i['id']),'file_name':i['file_name'],'clip':Path(i['file_name']).parent.name}
                  for i in VAL_COCO['images']]).sort_values(['clip','file_name']).reset_index(drop=True)
meta['n']=meta.groupby('clip').cumcount()
val_table=meta.loc[meta['n']%3==0].copy()
VAL_IDS=val_table['id'].astype(int).tolist()
clips=sorted(meta['clip'].unique())
assert len(clips)==7 and len(VAL_IDS)==951
TUNE_CLIPS=set(clips[:4]); CHECK_CLIPS=set(clips[4:]);assert TUNE_CLIPS.isdisjoint(CHECK_CLIPS)
TUNE_IDS=val_table.loc[val_table['clip'].isin(TUNE_CLIPS),'id'].astype(int).tolist()
CHECK_IDS=val_table.loc[val_table['clip'].isin(CHECK_CLIPS),'id'].astype(int).tolist()
print('Calibration:',len(TUNE_IDS),'images from',sorted(TUNE_CLIPS))
print('Unseen check:',len(CHECK_IDS),'images from',sorted(CHECK_CLIPS))

rt_f0 rt 66.2 MB
rt_f1 rt 66.2 MB
rt_f2 rt 66.2 MB
retina_final retina 291.3 MB
retina_f0 retina 291.3 MB
retina_f1 retina 291.3 MB
retina_f2 retina 291.3 MB
Dataset train 24201 val 2846
loading annotations into memory...
Done (t=0.85s)
creating index...
index created!
Calibration: 471 images from ['uav0000086_00000_v', 'uav0000117_02622_v', 'uav0000137_00458_v', 'uav0000182_00000_v']
Unseen check: 480 images from ['uav0000268_05773_v', 'uav0000305_00000_v', 'uav0000339_00001_v']


## 3 — Locate previous prediction caches; generate only missing ones

Old Kaggle caches are perfectly reusable if they refer to these *same* checkpoints. RT-DETR `lr2e4` and RetinaNet `resolution_768` use the same model labels as the previous `14_RTDETR_RetinaNet_Ensemble_Kaggle` notebook. For cached CV predictions, the code validates the image IDs and dimensions; **do not relabel caches from a different model or training run**.

If a cache is missing, inference can be performed from saved checkpoints (Colab GPU required). No new training runs here.

In [4]:
# Auto-unpack old cached JSON.GZ files (once). Source may be a ZIP from Kaggle or folder from Mac.
CACHE_ZIP=DRIVE/'ensemble_cache.zip'
if CACHE_ZIP.is_file():
    with zipfile.ZipFile(CACHE_ZIP) as z:
        for name in z.namelist():
            if name.endswith('.json.gz') and not name.startswith('__MACOSX/'):
                target=CACHE/Path(name).name
                if not target.exists():
                    with z.open(name) as src, target.open('wb') as dst:shutil.copyfileobj(src,dst)
    print('Imported cache ZIP:',CACHE_ZIP)

def fingerprint(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while block:=f.read(8*1024*1024):h.update(block)
    return h.hexdigest()[:12]

def cache_path(prefix,label):
    # On-disk filename must carry the hash of the exact currently selected weights.
    digest=fingerprint(WEIGHTS[label][1])
    expected=CACHE/f'{prefix}_{label}_{digest}.json.gz'
    return expected if expected.is_file() else None

for label in WEIGHTS:
    print(label,'val cache',bool(cache_path('val_step3',label)),'test cache',bool(cache_path('test_all',label)))

Imported cache ZIP: /content/drive/MyDrive/UCU_AI_HW2/ensemble_cache.zip
rt_f0 val cache True test cache True
rt_f1 val cache True test cache True
rt_f2 val cache True test cache True
retina_final val cache False test cache False
retina_f0 val cache True test cache True
retina_f1 val cache True test cache True
retina_f2 val cache True test cache True


In [5]:
# Load full dataset images only if inference or optional training is needed.
DATA_ROOT=None

def ensure_data():
    global DATA_ROOT
    if DATA_ROOT is not None:return DATA_ROOT
    base=Path('/content/visdrone');base.mkdir(parents=True,exist_ok=True)
    found=[p.parent.parent for p in base.rglob('train/annotations_coco.json')
           if (p.parent.parent/'val/images').is_dir() and (p.parent.parent/'test/images').is_dir()]
    if not found:
        with zipfile.ZipFile(COMPETITION_ZIP) as z:
            required=sum(m.file_size for m in z.infolist())
            free=shutil.disk_usage('/content').free
            assert free>required*1.12, f'Not enough disk: need {required/1e9:.1f}GB, have {free/1e9:.1f}GB'
            resolved=base.resolve()
            for m in z.infolist():
                dest=(base/m.filename).resolve()
                assert dest==resolved or resolved in dest.parents, f'Unsafe ZIP path {m.filename}'
            print('Extracting dataset once...')
            z.extractall(base)
        found=[p.parent.parent for p in base.rglob('train/annotations_coco.json')
               if (p.parent.parent/'val/images').is_dir() and (p.parent.parent/'test/images').is_dir()]
    assert len(found)==1,found
    DATA_ROOT=found[0]
    return DATA_ROOT

def image_index(split):
    root=ensure_data()/split/'images'
    files=[p for p in root.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png')]
    out={}
    for p in files:
        key=(p.parent.name,p.stem)
        assert key not in out,f'Duplicate key {key}'
        out[key]=p
    return out

def val_paths():
    index=image_index('val');paths={}
    for row in val_table.itertuples():
        f=Path(row.file_name);key=(f.parent.name,f.stem)
        assert key in index,key
        paths[str(int(row.id))]=index[key]
    return paths

def test_paths():
    index=image_index('test')
    paths={f'{clip}_{stem}':p for (clip,stem),p in index.items()}
    assert len(paths)==3811,len(paths)
    return dict(sorted(paths.items()))

## 4 — Load / compute raw detections (model by model, checkpoint-aware)

**Important:** cache re-use is automatic only for labels that correspond to the *same saved CV checkpoints*. The `retina_final` label is **new** and must be computed from the final saved RetinaNet weights. Checkpoint-specific cache files are generated for it.

In [6]:
MIN_CONF=.001; MAX_RAW_DETS=300; RT_BATCH=2

def build_retina():
    m=retinanet_resnet50_fpn_v2(weights=None,weights_backbone=None,min_size=768,
       max_size=1280,score_thresh=MIN_CONF,nms_thresh=.5,detections_per_img=MAX_RAW_DETS,topk_candidates=1000)
    h=m.head.classification_head;old=h.cls_logits
    counts=m.anchor_generator.num_anchors_per_location();assert len(set(counts))==1
    h.cls_logits=torch.nn.Conv2d(old.in_channels,counts[0]*8,3,stride=1,padding=1)
    h.num_classes=8
    return m

def load_retina(path):
    payload=torch.load(path,map_location='cpu',weights_only=True)
    assert isinstance(payload,dict) and 'model' in payload
    model=build_retina();model.load_state_dict(payload['model'],strict=True)
    return model.eval().to(DEVICE)


# Optional third architecture: Faster R-CNN (two-stage, COCO ids 1..8).
def build_faster():
    from torchvision.models.detection import fasterrcnn_resnet50_fpn
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    model=fasterrcnn_resnet50_fpn(
        weights=None,weights_backbone=None,min_size=768,max_size=1280,
        box_score_thresh=.001,box_nms_thresh=.5,box_detections_per_img=300,
        rpn_pre_nms_top_n_test=1000,rpn_post_nms_top_n_test=500)
    features=model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor=FastRCNNPredictor(features,9)
    return model

def load_faster(path):
    state=torch.load(path,map_location='cpu',weights_only=True)
    assert isinstance(state,dict) and 'model' in state
    model=build_faster()
    model.load_state_dict(state['model'],strict=True)
    return model.eval().to(DEVICE)

def compute_cache(label,split,paths):
    assert DEVICE.startswith('cuda'),'GPU required for uncached inference.'
    kind,ckpt=WEIGHTS[label]
    dest=CACHE/f'{split}_{label}_{fingerprint(ckpt)}.json.gz'
    if dest.exists():return dest
    res={}
    if kind=='retina':
        model=load_retina(ckpt)
        try:
            with torch.inference_mode():
                for id_,path in tqdm(paths.items(),desc=f'{label} {split}'):
                    with Image.open(path) as im:
                        width,height=im.size
                        x=pil_to_tensor(im.convert('RGB')).float().div_(255).to(DEVICE)
                    det=model([x])[0]
                    out=[]
                    for cls,score,box in zip(det['labels'].detach().cpu().tolist(),
                                             det['scores'].detach().cpu().tolist(),
                                             det['boxes'].detach().cpu().tolist()):
                        if 0<=int(cls)<8 and float(score)>=MIN_CONF:
                            x1,y1,x2,y2=map(float,box)
                            x1,x2=np.clip([x1,x2],0,width);y1,y2=np.clip([y1,y2],0,height)
                            if x2>x1 and y2>y1:out.append([int(cls),float(score),float(x1),float(y1),float(x2),float(y2)])
                    res[str(id_)]={'size':[width,height],'detections':out[:MAX_RAW_DETS]}
                    del x,det
        finally:
            del model;gc.collect();torch.cuda.empty_cache()
    elif kind=='faster':
        model=load_faster(ckpt)
        try:
            with torch.inference_mode():
                for id_,path in tqdm(paths.items(),desc=f'{label} {split}'):
                    with Image.open(path) as im:
                        width,height=im.size
                        x=pil_to_tensor(im.convert('RGB')).float().div_(255).to(DEVICE)
                    det=model([x])[0];out=[]
                    for cls,score,box in zip(det['labels'].detach().cpu().tolist(),
                                              det['scores'].detach().cpu().tolist(),
                                              det['boxes'].detach().cpu().tolist()):
                        if not 1<=int(cls)<=8 or float(score)<MIN_CONF:continue
                        x1,y1,x2,y2=map(float,box)
                        x1,x2=np.clip([x1,x2],0,width);y1,y2=np.clip([y1,y2],0,height)
                        if x2>x1 and y2>y1:out.append([int(cls)-1,float(score),x1,y1,x2,y2])
                    res[str(id_)]={'size':[width,height],'detections':out[:MAX_RAW_DETS]}
                    del det,x
        finally:
            del model;gc.collect();torch.cuda.empty_cache()
    elif kind=='rt':
        model=RTDETR(str(ckpt));items=list(paths.items())
        try:
            for start in tqdm(range(0,len(items),RT_BATCH),desc=f'{label} {split}'):
                batch=items[start:start+RT_BATCH]
                pred=model.predict(source=[str(p) for _,p in batch],imgsz=640,device=0,
                                   batch=RT_BATCH,conf=MIN_CONF,iou=.7,max_det=MAX_RAW_DETS,
                                   verbose=False,save=False,augment=False)
                assert len(pred)==len(batch)
                for (id_,_),out in zip(batch,pred):
                    height,width=out.orig_shape
                    dets=[]
                    if out.boxes is not None:
                        for box,cls,score in zip(out.boxes.xyxy.cpu().tolist(),
                                                 out.boxes.cls.int().cpu().tolist(),
                                                 out.boxes.conf.cpu().tolist()):
                            x1,y1,x2,y2=map(float,box)
                            x1,x2=np.clip([x1,x2],0,width);y1,y2=np.clip([y1,y2],0,height)
                            if int(cls) in range(8) and x2>x1 and y2>y1:
                                dets.append([int(cls),float(score),x1,y1,x2,y2])
                    res[str(id_)]={'size':[width,height],'detections':dets}
        finally:
            del model;gc.collect();torch.cuda.empty_cache()
    else:raise ValueError(kind)
    tmp=dest.with_suffix('.partial')
    with gzip.open(tmp,'wt') as f:json.dump(res,f)
    tmp.replace(dest)
    print('Cached',dest,'size MB',round(dest.stat().st_size/1e6,1))
    return dest

def load_predictions(label,split,expected_ids):
    f=cache_path(split,label)
    if f is None:
        paths=val_paths() if split=='val_step3' else test_paths()
        f=compute_cache(label,split,paths)
    with gzip.open(f,'rt') as g:raw=json.load(g)
    expected=set(map(str,expected_ids))
    assert set(raw)==expected,(label,split,len(raw),len(expected),list(set(raw)^expected)[:5])
    for sample in list(raw.values())[:3]:
        assert len(sample['size'])==2 and 'detections' in sample
    return raw

# If you uploaded historical caches, validation inference is skipped for rt_f* and retina_f*.
# Full RetinaNet inference is run once when retina_final is available.
VAL_RAW={}
for name in WEIGHTS:
    VAL_RAW[name]=load_predictions(name,'val_step3',VAL_IDS)
print('Holdout predictions loaded:',list(VAL_RAW))

Extracting dataset once...


retina_final val_step3:   0%|          | 0/951 [00:00<?, ?it/s]

Cached /content/drive/MyDrive/UCU_AI_HW2/ensemble_cache/val_step3_retina_final_96554ce72d02.json.gz size MB 9.3
Holdout predictions loaded: ['rt_f0', 'rt_f1', 'rt_f2', 'retina_final', 'retina_f0', 'retina_f1', 'retina_f2']


## 5 — Tune **fusion**, not training epochs, on four validation videos

Original EDA: approximately **45 objects per frame**, **24.4% small objects**, dense crowds and many `iscrowd` regions. Fusion IoU/score ranking can materially affect *small neighboring objects* and the COCO maxDets=100 metric. We test modest, prespecified alternatives rather than selecting dozens of class-specific parameters that would overfit.

The previous Kaggle ensemble used `IoU=0.55`, RetinaNet weight 1.5× and score penalty for unmatched detections. This experiment tries more conservative overlap thresholds, calibrated score penalties and the full-training RetinaNet.

In [7]:
def iou(a,b):
    u=max(0.,min(a[2],b[2])-max(a[0],b[0]))*max(0.,min(a[3],b[3])-max(a[1],b[1]))
    area=lambda x:max(0,x[2]-x[0])*max(0,x[3]-x[1])
    return u/(area(a)+area(b)-u+1e-9)

def weights_for(rt_total=1,ret_total=1.5,use_final=True,faster_total=0.):
    rt=[f'rt_f{i}' for i in range(3)]
    if use_final and 'retina_final' in VAL_RAW:
        retina=['retina_final']
    else:
        retina=[f'retina_f{i}' for i in range(3)]
    assert all(x in VAL_RAW for x in rt+retina)
    result={**{x:rt_total/len(rt) for x in rt},
            **{x:ret_total/len(retina) for x in retina}}
    if faster_total>0:
        faster=[f'faster_f{i}' for i in range(3)]
        assert all(x in VAL_RAW for x in faster), 'Faster checkpoints not loaded'
        result.update({x:faster_total/len(faster) for x in faster})
    return result

def fuse_one(raw,image_id,weights,iou_thr=.55,coverage_power=1.0,max_det=300,
             score_floor=.001):
    image_id=str(image_id)
    # Sources predict COCO classes 0..7; never fuse boxes across different categories.
    sizes={tuple(raw[src][image_id]['size']) for src in weights}
    assert len(sizes)==1,(image_id,sizes)
    total=sum(weights.values())
    groups=defaultdict(list)
    for src,weight in weights.items():
        for cls,score,x1,y1,x2,y2 in raw[src][image_id]['detections']:
            if 0<=int(cls)<8 and score>=score_floor and x2>x1 and y2>y1:
                groups[int(cls)].append((src,float(weight),float(score),[x1,y1,x2,y2]))
    merged=[]
    for cls,items in groups.items():
        clusters=[]
        for src,w,score,box in sorted(items,key=lambda k:k[1]*k[2],reverse=True):
            best_idx=None;best_iou=iou_thr
            for idx,cl in enumerate(clusters):
                if src in cl['sources']:continue
                v=iou(box,cl['box'])
                if v>best_iou:best_idx=idx;best_iou=v
            if best_idx is None:clusters.append({'sources':{src},'box':box,'members':[(w,score,box)]})
            else:
                cl=clusters[best_idx];cl['sources'].add(src);cl['members'].append((w,score,box))
                cl['box']=np.average([m[2] for m in cl['members']],
                         weights=[m[0]*m[1] for m in cl['members']],axis=0).tolist()
        for cl in clusters:
            w_present=sum(m[0] for m in cl['members'])
            avg_score=sum(m[0]*m[1] for m in cl['members'])/w_present
            # coverage_power=1 matches previous notebook; 0.5 penalizes singleton less.
            coverage=(w_present/total)**coverage_power
            score=min(1.,avg_score*coverage)
            x1,y1,x2,y2=cl['box']
            if score>=score_floor and x2>x1 and y2>y1:
                merged.append([cls,score,x1,y1,x2,y2])
    return sorted(merged,key=lambda r:r[1],reverse=True)[:max_det]

def to_coco(raw,ids,config):
    w=config['weights'];det=[]
    for image_id in ids:
        for cls,score,x1,y1,x2,y2 in fuse_one(raw,image_id,w,
            iou_thr=config['iou'],coverage_power=config['coverage'],max_det=config['max_det']):
            det.append({'image_id':int(image_id),'category_id':int(cls)+1,
                        'bbox':[x1,y1,x2-x1,y2-y1],'score':float(score)})
    return det

def evaluate(det,ids,verbose=False):
    assert det,'No detections in COCO evaluation'
    dt=GT.loadRes(det)
    e=COCOeval(GT,dt,iouType='bbox')
    e.params.imgIds=list(map(int,ids));e.params.catIds=list(range(1,9));e.params.maxDets=[1,10,100]
    if not verbose:
        import contextlib
        with contextlib.redirect_stdout(io.StringIO()):e.evaluate();e.accumulate();e.summarize()
    else:e.evaluate();e.accumulate();e.summarize()
    s=e.stats
    return dict(mAP=float(s[0]),AP50=float(s[1]),AP75=float(s[2]),
                AP_small=float(s[3]),AP_medium=float(s[4]),AP_large=float(s[5]))

# Small predeclared search: no 100s of permutations to avoid overfitting holdout.
CONFIGS=[]
for source in ['cv','final']:
    if source=='final' and 'retina_final' not in VAL_RAW:continue
    if source=='cv' and not all(f'retina_f{i}' in VAL_RAW for i in range(3)):continue
    for title,ratio,thr,power,cap in [
        ('legacy',1.5,.55,1.,300),
        ('relaxed',1.5,.55,.5,300),
        ('tight',1.5,.65,1.,300),
        ('loose',1.5,.45,1.,300),
        ('balanced',1.0,.55,.5,300),
        ('rt_favor',.75,.55,.5,300),
        ('top100',1.5,.55,.5,100)]:
        CONFIGS.append({'name':f'{source}_{title}',
                'weights':weights_for(1.,ratio,use_final=(source=='final')),
                'iou':thr,'coverage':power,'max_det':cap})
if INCLUDE_FASTER:
    for source in ['cv','final']:
        if source=='final' and 'retina_final' not in VAL_RAW:continue
        if source=='cv' and not all(f'retina_f{i}' in VAL_RAW for i in range(3)):continue
        for extra in (.5,1.):
            CONFIGS.append({'name':f'{source}_faster_{extra}',
               'weights':weights_for(1.,1.5,use_final=(source=='final'),faster_total=extra),
               'iou':.55,'coverage':.5,'max_det':300})
print('Calibration candidates:',len(CONFIGS))

results=[]
for config in CONFIGS:
    det=to_coco(VAL_RAW,TUNE_IDS,config)
    m=evaluate(det,TUNE_IDS)
    results.append({'name':config['name'],**m})
tuning=pd.DataFrame(results).sort_values('mAP',ascending=False)
display(tuning.round(5))
tuning.to_csv(OUT/'holdout_tuning.csv',index=False)
SELECTED_CONFIG=next(c for c in CONFIGS if c['name']==tuning.iloc[0]['name'])
print('Chosen from tuning video clips only:',SELECTED_CONFIG['name'])

Calibration candidates: 14
Loading and preparing results...
DONE (t=0.20s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.19s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.19s)
creating index...
index created!
Loading and preparing results...
DONE (t=2.12s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.20s)
creating index...
index created!
Loading and preparing results...
DONE (t=2.25s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.07s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.19s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.65s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.31s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.81s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.29s)
creating index...
index created!
L

,name,mAP,AP50,AP75,AP_small,AP_medium,AP_large
11,final_balanced,0.25596,0.52012,0.21967,0.13076,0.23487,0.42091
8,final_relaxed,0.25525,0.51686,0.22029,0.12591,0.23318,0.42131
12,final_rt_favor,0.25469,0.52090,0.21638,0.13296,0.23456,0.41873
7,final_legacy,0.25295,0.51301,0.21722,0.12633,0.23292,0.42074
10,final_loose,0.25243,0.51448,0.21551,0.12592,0.23215,0.41971
9,final_tight,0.25234,0.50942,0.21896,0.12544,0.23264,0.42177
1,cv_relaxed,0.24977,0.50388,0.21699,0.12549,0.23301,0.40955
4,cv_balanced,0.24908,0.50357,0.21547,0.12828,0.23365,0.41020
0,cv_legacy,0.24725,0.49808,0.21467,0.12284,0.23210,0.41182
3,cv_loose,0.24661,0.49979,0.21193,0.12230,0.23079,0.41051


Chosen from tuning video clips only: final_balanced


## 6 — Independent holdout video check + compare with the original ensemble

These **three clips were not used** to select the fusion hyperparameters. Compare candidate to the previous `cv_legacy` and simple RT-only / Retina-only options; **do not retune** after examining them. A small improvement on 3/7 videos does **not** guarantee better public or private Kaggle score.

In [8]:
CHECK_CONFIGS=[SELECTED_CONFIG]
legacy=next((c for c in CONFIGS if c['name']=='cv_legacy'),None)
if legacy and legacy['name']!=SELECTED_CONFIG['name']:CHECK_CONFIGS.append(legacy)
if 'retina_final' in VAL_RAW:
    CHECK_CONFIGS.append({'name':'retina_final_only','weights':{'retina_final':1.},
                          'iou':.55,'coverage':0.,'max_det':300})
CHECK_CONFIGS.append({'name':'rt_only','weights':{f'rt_f{i}':1/3 for i in range(3)},
                     'iou':.55,'coverage':0.,'max_det':300})
check=[]
for cfg in CHECK_CONFIGS:
    m=evaluate(to_coco(VAL_RAW,CHECK_IDS,cfg),CHECK_IDS)
    check.append({'name':cfg['name'],**m})
checked=pd.DataFrame(check).sort_values('mAP',ascending=False)
display(checked.round(5))
checked.to_csv(OUT/'holdout_check.csv',index=False)

# Honest full-holdout metric, for record only. Config stays fixed.
full=evaluate(to_coco(VAL_RAW,VAL_IDS,SELECTED_CONFIG),VAL_IDS)
print('Selected full 951-frame holdout:',SELECTED_CONFIG['name'],full)
(OUT/'selected_fusion_config.json').write_text(json.dumps(SELECTED_CONFIG,indent=2))

Loading and preparing results...
DONE (t=0.22s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.82s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.26s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.47s)
creating index...
index created!


,name,mAP,AP50,AP75,AP_small,AP_medium,AP_large
1,cv_legacy,0.20580,0.34278,0.21403,0.06959,0.21507,0.43338
0,final_balanced,0.19689,0.33451,0.20035,0.07042,0.20086,0.43491
2,retina_final_only,0.16509,0.28096,0.16848,0.04606,0.17437,0.41320
3,rt_only,0.15137,0.26839,0.14330,0.06591,0.14594,0.37657


Loading and preparing results...
DONE (t=0.39s)
creating index...
index created!
Selected full 951-frame holdout: final_balanced {'mAP': 0.20908359232445425, 'AP50': 0.40871807800624094, 'AP75': 0.18606875461068095, 'AP_small': 0.09233098144058713, 'AP_medium': 0.1922384323563282, 'AP_large': 0.35847060779623374}


225

## 7 — Produce a new Kaggle submission using the selected fusion

This stage uses **test images only for inference**. If previous `.json.gz` test predictions are on Drive, RT-DETR and historical RetinaNet inference will be skipped. **Final RetinaNet** predictions (if used) must be generated once from its `best.pt`; no training occurs here. Output is persisted directly on Drive.

**Potentially long GPU cell:** test inference for 3 RT folds + 1 final RetinaNet if caches are missing. Run only when you are ready; Colab disconnects can interrupt the current model, but previously completed model caches stay on Drive.

In [9]:
TEST_RAW={}
SELECTED_LABELS=list(SELECTED_CONFIG['weights'])
for label in SELECTED_LABELS:
    test_ids=list(test_paths()) if cache_path('test_all',label) is None else None
    if test_ids is None:
        # Recover IDs from an existing cache; avoid extracting 11GB just for this.
        with gzip.open(cache_path('test_all',label),'rt') as g:
            # For first time only; model results are loaded below.
            cached=json.load(g)
        test_ids=list(cached)
        del cached
    TEST_RAW[label]=load_predictions(label,'test_all',test_ids)
# Prefer the competition sample submission ordering if available, else sort stable file IDs.
all_ids=set(next(iter(TEST_RAW.values())))
assert len(all_ids)==3811 and all(set(v)==all_ids for v in TEST_RAW.values())
TEST_IDS=sorted(all_ids)
print('Loaded all test predictions for',SELECTED_LABELS)

retina_final test_all:   0%|          | 0/3811 [00:00<?, ?it/s]

Cached /content/drive/MyDrive/UCU_AI_HW2/ensemble_cache/test_all_retina_final_96554ce72d02.json.gz size MB 38.4
Loaded all test predictions for ['rt_f0', 'rt_f1', 'rt_f2', 'retina_final']


In [10]:
rows=[]
for id_ in tqdm(TEST_IDS,desc='Building updated Kaggle submission'):
    dets=fuse_one(TEST_RAW,id_,SELECTED_CONFIG['weights'],
                  iou_thr=SELECTED_CONFIG['iou'],
                  coverage_power=SELECTED_CONFIG['coverage'],
                  max_det=SELECTED_CONFIG['max_det'])
    tok=[]
    for cls,score,x1,y1,x2,y2 in dets:
        if 0<=int(cls)<8 and 0<=score<=1 and x2>x1 and y2>y1:
            tok.extend([str(cls),f'{score:.6f}',f'{x1:.2f}',f'{y1:.2f}',
                        f'{x2-x1:.2f}',f'{y2-y1:.2f}'])
    rows.append({'image_id':str(id_),'PredictionString':' '.join(tok)})
submission=pd.DataFrame(rows)
assert len(submission)==3811 and submission.image_id.nunique()==3811
assert submission.columns.tolist()==['image_id','PredictionString']
assert all(len(x.split())%6==0 for x in submission.PredictionString)
SUBMISSION=OUT/'submission.csv'
submission.to_csv(SUBMISSION,index=False)
print('SAVED:',SUBMISSION,'MB',round(SUBMISSION.stat().st_size/1e6,1))
display(submission.head())

Building updated Kaggle submission:   0%|          | 0/3811 [00:00<?, ?it/s]

SAVED: /content/drive/MyDrive/UCU_AI_HW2/ensemble15_final/submission.csv MB 42.6


,image_id,PredictionString
0,clip_001_0000001,0 0.825999 778.39 887.12 64.78 173.05 0 0.7935...
1,clip_001_0000002,0 0.824258 716.37 897.57 65.31 177.53 0 0.8121...
2,clip_001_0000003,0 0.809845 709.85 898.94 65.78 177.61 0 0.8027...
3,clip_001_0000004,0 0.841758 757.47 902.17 68.18 172.94 0 0.8045...
4,clip_001_0000005,0 0.819965 1662.61 806.55 56.46 153.21 0 0.792...


## 8 — Backup in Google Drive (do not lose files after disconnect)

The ZIP includes the new submission, COCOeval calibration tables and fusion config; it does **not** duplicate multi-GB model checkpoints or raw test prediction caches. Those already remain in `UCU_AI_HW2`.

In [11]:
BUNDLE=OUT/'submission_bundle.zip'
with zipfile.ZipFile(BUNDLE,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in [SUBMISSION,OUT/'holdout_tuning.csv',OUT/'holdout_check.csv',OUT/'selected_fusion_config.json']:
        assert p.is_file(),p
        z.write(p,arcname=p.name)
with zipfile.ZipFile(BUNDLE) as z:assert z.testzip() is None
print('SUBMISSION READY:',SUBMISSION)
print('BACKUP READY:',BUNDLE,'MB',round(BUNDLE.stat().st_size/1e6,1))

SUBMISSION READY: /content/drive/MyDrive/UCU_AI_HW2/ensemble15_final/submission.csv
BACKUP READY: /content/drive/MyDrive/UCU_AI_HW2/ensemble15_final/submission_bundle.zip MB 17.4


## 9 — OPTIONAL: RT-DETR fine-tune with all 56 train videos, more epochs, and checkpoint saves

**This section DOES train**. Default **off**. Use this only if you have substantial GPU budget and the cached-ensemble improvements are not enough. It initializes from the **already trained fold-0** `lr2e4` checkpoint, so it is **additional fine-tuning, not training from scratch**. Since the final training set includes all 56 training videos (every sixth frame here to reduce cost), it can improve generalization but also overfit. A small LR (`3e-5`), validation and early stopping limit that risk; they cannot eliminate it.

- `RT_EXTRA_EPOCHS=4`, `batch=2`, `imgsz=640`, `mosaic=0`, `amp=False`, `patience=2`, optimizer AdamW.
- Saves Ultralytics `best.pt`/`last.pt` **directly to Google Drive**. These weights are **not automatically included in the submission made in section 7**. To test the new RT model, explicitly register it as `rt_final`, recompute its validation predictions, then rerun section 5 onward. Never silently use a new checkpoint with old cached predictions.
- Do **not** train RT-DETR if the existing ensemble already meets your goal. This section can take many GPU hours.

In [12]:
RUN_RT_FINETUNE=False
RT_EXTRA_EPOCHS=4
RT_FIN_DIR=DRIVE/'rtdetr_finetune_full56'
RT_FIN_DIR.mkdir(parents=True,exist_ok=True)

def train_rtdetr_extra():
    assert DEVICE.startswith('cuda'),'GPU required.'
    data=ensure_data()
    # Exact original per-video sample with chronological frame_step=6.
    train_meta=pd.DataFrame([{'file_name':i['file_name'],
                              'clip':Path(i['file_name']).parent.name} for i in TRAIN_COCO['images']])
    train_meta=train_meta.sort_values(['clip','file_name']).reset_index(drop=True)
    train_meta['index']=train_meta.groupby('clip').cumcount()
    selected=train_meta.loc[train_meta['index']%6==0]
    val_meta=meta.loc[meta['n']%6==0]
    assert selected['clip'].nunique()==56 and val_meta['clip'].nunique()==7
    def locate(split,rel):
        rel=Path(rel);root=data/split
        options=[root/rel,root/'images'/rel,root/'images'/rel.parent.name/rel.name]
        found=[p for p in options if p.is_file()]
        assert len(found)>=1, f'Missing {split} image: {rel}'
        return found[0]
    trains=[locate('train',f) for f in selected['file_name']]
    vals=[locate('val',f) for f in val_meta['file_name']]
    for split,images in [('train',trains),('val',vals)]:
        # YOLO dataset already has matching labels/{clip}/{frame}.txt in the competition ZIP.
        for p in images[:20]:
            lab=Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')
            assert lab.is_file(),f'YOLO label missing: {lab}'
        (RT_FIN_DIR/f'{split}.txt').write_text('\n'.join(map(str,images))+'\n')
    # YAML path lists with image/label sibling directory structure.
    import yaml
    yaml_path=RT_FIN_DIR/'dataset.yaml'
    yaml.safe_dump({'path':str(data),'train':str(RT_FIN_DIR/'train.txt'),
                    'val':str(RT_FIN_DIR/'val.txt'),'nc':8,
                    'names':{i:c for i,c in enumerate(['person','bicycle','car','van','truck','tricycle','bus','motor'])}},
                    yaml_path.open('w'),sort_keys=False)
    run_dir=RT_FIN_DIR/'full56'
    last=run_dir/'weights/last.pt'
    model=RTDETR(str(last if last.is_file() else RT_WEIGHTS[0]))
    if last.is_file():
        print('Resuming previous fine-tune using optimizer state in last.pt')
        model.train(resume=True,device=0,workers=0)
    else:
        model.train(data=str(yaml_path),epochs=RT_EXTRA_EPOCHS,imgsz=640,batch=2,
                    device=0,workers=0,optimizer='AdamW',lr0=3e-5,lrf=.2,
                    weight_decay=1e-4,warmup_epochs=1,cos_lr=True,
                    mosaic=0.,amp=False,seed=42,deterministic=False,
                    patience=2,save=True,save_period=1,plots=False,
                    project=str(RT_FIN_DIR),name='full56',exist_ok=True)
    assert (run_dir/'weights/best.pt').is_file()
    print('Saved fine-tuned RT-DETR best:',run_dir/'weights/best.pt')
    print('This model must be evaluated as a NEW source. Do not reuse rt_f0 cache.')
    return run_dir/'weights/best.pt'

if RUN_RT_FINETUNE:
    rt_final_path=train_rtdetr_extra()
    # To evaluate the new model, manually register it and rerun validation+fusion:
    WEIGHTS['rt_final']=('rt',rt_final_path)
    print('NEW SOURCE registered as rt_final. Recompute its validation/test caches before use.')
else:
    print('Optional RT-DETR fine-tuning skipped. Current submission uses already-trained checkpoints.')

Optional RT-DETR fine-tuning skipped. Current submission uses already-trained checkpoints.


result
0.2120148373
